# Lesson 4 — Research Tool Agent (Live Web Search) → A2A Server → A2A Client

**Audience:** University faculty (Malaysia)  
**Goal:** Build a **research agent** that uses **live web search** with citations, then wrap it as an **A2A server** and test via an **A2A client**.

We will use **OpenAI Responses API + the built-in `web_search` tool**:
- `tools=[{"type":"web_search"}]`
- `include=["web_search_call.action.sources"]` to receive source metadata
- enforce a strict output format: **Answer / Sources / Notes**

> This lesson is intentionally simple: one agent, one tool, one response.
> In Lesson 5, the Orchestrator will route “Not found” policy questions to this Research agent.

---

## Can we build this with CrewAI?
**Yes**, but most CrewAI web-search tools (Tavily / Serper / Brave / Exa) require extra API keys.  
Since you only want to use your **OpenAI API key**, we implement a **custom CrewAI tool** that calls OpenAI’s built-in `web_search`.  
(We keep CrewAI as an **optional** section so the lesson remains stable for everyone.)


## 1. Setup

### 1.1 Install dependencies (run once)

Required:
- `openai` (Responses API)
- `a2a-sdk[http-server]` (A2A server)
- `httpx` (A2A client)
- `python-dotenv`

Optional (CrewAI variant):
- `crewai` and `crewai-tools`


In [ ]:
# If needed, uncomment and run:
!pip -q install openai "a2a-sdk[http-server]" httpx python-dotenv

# Optional CrewAI variant:
!pip -q install crewai crewai-tools

# takes around 2-5 minutes to install, depending on your system and network speed

In [2]:
import os
from dotenv import load_dotenv

_ = load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Please set OPENAI_API_KEY in your environment."
print("✅ OPENAI_API_KEY found")

✅ OPENAI_API_KEY found


## 2. Quick demo: OpenAI built-in Web Search tool

Minimal pattern:

```python
resp = client.responses.create(
  model="gpt-5",
  tools=[{"type":"web_search"}],
  include=["web_search_call.action.sources"],
  input="Your question..."
)
```

If your model access differs, try switching:
- `gpt-4o-mini` (often available)
- `o4-mini` or `gpt-5` (as documented for web_search)


In [3]:
from openai import OpenAI
import json

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

MODEL = "gpt-4o-mini"  # if web_search isn't supported, try "o4-mini" or "gpt-5"

def extract_web_sources(resp) -> list[dict]:
    """Best-effort extraction of web search sources from a Responses API response."""
    sources = []
    output = getattr(resp, "output", None) or []
    for item in output:
        t = getattr(item, "type", None) or (item.get("type") if isinstance(item, dict) else None)
        if t == "web_search_call":
            action = getattr(item, "action", None) or (item.get("action") if isinstance(item, dict) else None)
            if action is None:
                continue
            s = getattr(action, "sources", None) or (action.get("sources") if isinstance(action, dict) else None)
            if s:
                for src in s:
                    if isinstance(src, dict):
                        sources.append(src)
                    else:
                        # pydantic-like
                        sources.append(getattr(src, "model_dump", lambda: {"title": str(src)})())
    return sources

q = "What are recent academic integrity guidelines about using AI tools in coursework? Provide key points."
resp = client.responses.create(
    model=MODEL,
    tools=[{"type": "web_search"}],
    include=["web_search_call.action.sources"],
    input=q,
)

print("=== Model answer (text) ===")
print((resp.output_text or "")[:800])

sources = extract_web_sources(resp)
print("\n=== Sources (first 5) ===")
print(json.dumps(sources[:5], indent=2)[:1200])

=== Model answer (text) ===
Recent academic integrity guidelines emphasize responsible use of AI tools in coursework, focusing on transparency, proper attribution, and critical engagement. Key points include:

1. **Clear Communication of Policies**: Instructors should explicitly state their AI usage policies in syllabi and assignment instructions, detailing when and how AI tools are permitted. ([teaching.cornell.edu](https://teaching.cornell.edu//generative-artificial-intelligence/ai-academic-integrity?utm_source=openai))

2. **Transparency and Attribution**: Students must disclose any AI assistance in their work and provide appropriate citations, similar to other sources. ([fit.edu](https://www.fit.edu/policies/student-focused-policies/standards-and-policies/responsible-use-of-generative-ai-in-academic-work/?utm_sou

=== Sources (first 5) ===
[
  {
    "type": "url",
    "url": "https://teaching.cornell.edu//generative-artificial-intelligence/ai-academic-integrity?utm_source=openai"
 

## 3. Research function (strict formatting + citations)

We wrap the pattern into a function that guarantees an output format:

**Answer:**  
**Sources:** (3–6)  
**Notes:**


In [4]:
SYSTEM_RULES = """You are a research assistant for university faculty.

STRICT RULES:
1) Use web_search to find up-to-date information.
2) Your final output MUST include a Sources section with 3–6 sources.
3) Each source must include title and URL (if available from tool).
4) Do not invent citations. If sources are not available, say so explicitly.
5) Output exactly in this format:

Answer:
- ...

Sources:
- <Title> — <URL>
- ...

Notes:
- ...
"""

def research_with_citations(question: str) -> str:
    resp = client.responses.create(
        model=MODEL,
        tools=[{"type": "web_search"}],
        include=["web_search_call.action.sources"],
        input=[
            {"role": "system", "content": SYSTEM_RULES},
            {"role": "user", "content": question},
        ],
        temperature=0.2,
    )

    sources = extract_web_sources(resp)
    text = (resp.output_text or "").strip()

    if "Sources:" not in text and sources:
        lines = [text, "", "Sources:"]
        for s in sources[:6]:
            title = s.get("title") or s.get("name") or "Source"
            url = s.get("url") or s.get("link") or ""
            lines.append(f"- {title}" + (f" — {url}" if url else ""))
        lines += ["", "Notes:", "- (Auto-added sources from tool metadata.)"]
        text = "\n".join(lines)

    return text

from IPython.display import Markdown, display
display(Markdown(research_with_citations(
    "List 4 practical classroom policies universities publish for generative AI usage (allowed vs disallowed), with sources."
)))

Answer:
Universities are implementing various policies to guide the use of generative AI tools in classroom settings. Here are four practical classroom policies:

1. **University of Wisconsin-Stevens Point (UWSP): Generative AI Classroom Usage Policy**
   - **Policy Overview**: UWSP prohibits the use of generative AI for completing graded coursework, such as final exams or projects, unless explicitly permitted by an instructor. Faculty are encouraged to develop clear policies regarding AI usage, specifying allowed tools, purposes, and acknowledgment methods.
   - **Source**: ([handbook.uwsp.edu](https://handbook.uwsp.edu/pages/kRMszUJZrtxUG8MYL8Py?utm_source=openai))

2. **University of North Alabama (UNA): Generative AI Policy**
   - **Policy Overview**: UNA emphasizes responsible AI use, advising users to select AI tools thoughtfully and verify the accuracy of AI-generated content before academic use. Faculty are encouraged to include an AI policy statement in each course syllabus, be transparent about AI-generated content, and discourage reliance on AI-detection software for verifying originality.
   - **Source**: ([una.edu](https://www.una.edu/academics/generative-ai-policy.html?utm_source=openai))

3. **Harvard University: Generative AI Guidance**
   - **Policy Overview**: Harvard provides faculty with guidance on AI usage, offering sample policies ranging from prohibitive to fully encouraging approaches. Faculty are required to inform students of the policies governing generative AI use in class, with clear articulation in course syllabi.
   - **Source**: ([oue.fas.harvard.edu](https://oue.fas.harvard.edu/faculty-resources/generative-ai-guidance/?utm_source=openai))

4. **Princeton University: Generative AI and Our Classrooms**
   - **Policy Overview**: Princeton encourages faculty to experiment with generative AI tools, providing free access to Microsoft Copilot. Faculty are advised to articulate clear policies on AI use in syllabi, specifying permitted activities, disclosure requirements, and best practices for AI usage.
   - **Source**: ([mcgraw.princeton.edu](https://mcgraw.princeton.edu/generative-ai?utm_source=openai))

Sources:
- Generative Artificial Intelligence (AI) Classroom Usage Policy | University of Wisconsin-Stevens Point Calendar — ([handbook.uwsp.edu](https://handbook.uwsp.edu/pages/kRMszUJZrtxUG8MYL8Py?utm_source=openai))
- Generative AI Policy | University of North Alabama — ([una.edu](https://www.una.edu/academics/generative-ai-policy.html?utm_source=openai))
- Generative AI Guidance – Office of Undergraduate Education | Harvard University — ([oue.fas.harvard.edu](https://oue.fas.harvard.edu/faculty-resources/generative-ai-guidance/?utm_source=openai))
- Generative AI and Our Classrooms | Princeton University — ([mcgraw.princeton.edu](https://mcgraw.princeton.edu/generative-ai?utm_source=openai))

Notes:
- Policies regarding generative AI usage in academic settings are evolving. It's advisable to consult the latest guidelines from individual institutions for the most current information.
- The effectiveness of these policies depends on clear communication and consistent enforcement by faculty and administration.

## 4. Refactor into a reusable Agent class (`agents_research.py`)

In [5]:
%%writefile agents_research.py
from __future__ import annotations

import os
from typing import Any, Dict, List

from dotenv import load_dotenv
from openai import OpenAI


class WebResearchAgent:
    """Live web-search research agent using OpenAI's built-in web_search tool."""

    def __init__(self, model: str = "gpt-4o-mini") -> None:
        load_dotenv()
        if not os.environ.get("OPENAI_API_KEY"):
            raise RuntimeError("OPENAI_API_KEY is not set in the environment.")
        self.client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        self.model = model

        self.system_rules = """You are a research assistant for university faculty.

STRICT RULES:
1) Use web_search to find up-to-date information.
2) Your final output MUST include a Sources section with 3–6 sources.
3) Each source must include title and URL (if available from tool).
4) Do not invent citations. If sources are not available, say so explicitly.
5) Output exactly in this format:

Answer:
- ...

Sources:
- <Title> — <URL>
- ...

Notes:
- ...
"""

    def _extract_web_sources(self, resp: Any) -> List[Dict[str, Any]]:
        sources: List[Dict[str, Any]] = []
        output = getattr(resp, "output", None) or []
        for item in output:
            t = getattr(item, "type", None) or (item.get("type") if isinstance(item, dict) else None)
            if t == "web_search_call":
                action = getattr(item, "action", None) or (item.get("action") if isinstance(item, dict) else None)
                if action is None:
                    continue
                s = getattr(action, "sources", None) or (action.get("sources") if isinstance(action, dict) else None)
                if s:
                    for src in s:
                        if isinstance(src, dict):
                            sources.append(src)
                        else:
                            sources.append(getattr(src, "model_dump", lambda: {"title": str(src)})())
        return sources

    def answer_query(self, question: str) -> str:
        resp = self.client.responses.create(
            model=self.model,
            tools=[{"type": "web_search"}],
            include=["web_search_call.action.sources"],
            input=[
                {"role": "system", "content": self.system_rules},
                {"role": "user", "content": question},
            ],
            temperature=0.2,
        )

        text = (resp.output_text or "").strip()
        sources = self._extract_web_sources(resp)

        if "Sources:" not in text and sources:
            lines = [text, "", "Sources:"]
            for s in sources[:6]:
                title = s.get("title") or s.get("name") or "Source"
                url = s.get("url") or s.get("link") or ""
                lines.append(f"- {title}" + (f" — {url}" if url else ""))
            lines += ["", "Notes:", "- (Auto-added sources from tool metadata.)"]
            text = "\n".join(lines)

        return text


Writing agents_research.py


In [6]:
from agents_research import WebResearchAgent
from IPython.display import Markdown, display

research_agent = WebResearchAgent(model=MODEL)
display(Markdown(research_agent.answer_query(
    "What are 3 common principles in university AI policies (transparency, attribution, permitted use)? Provide sources."
)))

Universities commonly establish AI policies based on three key principles: transparency, attribution, and permitted use.

**Transparency**: Institutions emphasize the need for clear disclosure regarding AI usage. Users are required to inform stakeholders when AI tools are employed, detailing their application and purpose. This openness fosters trust and ensures ethical practices. ([aamc.org](https://www.aamc.org/about-us/mission-areas/medical-education/principles-ai-use/ensure-ethical-and-transparent-use?utm_source=openai))

**Attribution**: Proper acknowledgment of AI-generated content is mandated. Users must cite AI tools as they would any other source, specifying the tool's name, version, and usage context. This practice upholds academic integrity and intellectual property rights. ([price.university](https://price.university/ai-policy/?utm_source=openai))

**Permitted Use**: Universities define acceptable AI applications within academic settings. Policies specify scenarios where AI can be utilized, such as for research assistance or idea generation, while prohibiting its use in ways that compromise academic standards, like submitting AI-generated content as one's own work. ([law.georgetown.edu](https://www.law.georgetown.edu/ctls/wp-content/uploads/sites/33/2025/10/CTLS-Policy-on-the-Use-of-Artificial-Intelligence-in-Academic-Work.pdf?utm_source=openai))

**Sources**:
- Ensure Ethical and Transparent Use — https://www.aamc.org/about-us/mission-areas/medical-education/principles-ai-use/ensure-ethical-and-transparent-use
- AI Policy | Price University — https://price.university/ai-policy/
- Policy on the Use of Artificial Intelligence in Academic Work — https://www.law.georgetown.edu/ctls/wp-content/uploads/sites/33/2025/10/CTLS-Policy-on-the-Use-of-Artificial-Intelligence-in-Academic-Work.pdf

Sources:
- Source — https://ropercenter.cornell.edu/policies/roper-center-artificial-intelligence-ai-policy?utm_source=openai
- Source — https://teaching.cornell.edu/generative-artificial-intelligence/ai-course-policy-icons?utm_source=openai
- Source — https://www.american.edu/oit/security/Responsible-Use-of-AI.cfm?utm_source=openai
- Source — https://www.aamc.org/about-us/mission-areas/medical-education/principles-ai-use/ensure-ethical-and-transparent-use?utm_source=openai
- Source — https://www.unomaha.edu/artificial-intelligence/guiding-principles/faculty-guiding-principles.php?utm_source=openai
- Source — https://uona.edu/ai/policy/student-focus?utm_source=openai

Notes:
- (Auto-added sources from tool metadata.)

## 5. Wrap as an A2A Server (`a2a_web_research_agent.py`)

Run as:
```bash
python a2a_web_research_agent.py
```

Default URL: `http://127.0.0.1:9997/`


In [7]:
%%writefile a2a_web_research_agent.py
import os
import uvicorn
from dotenv import load_dotenv

from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.apps import A2AStarletteApplication
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import AgentCapabilities, AgentCard, AgentSkill
from a2a.utils import new_agent_text_message

from agents_research import WebResearchAgent


class WebResearchExecutor(AgentExecutor):
    def __init__(self) -> None:
        model = os.environ.get("RESEARCH_MODEL", "gpt-4o-mini")
        self.agent = WebResearchAgent(model=model)

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        prompt = context.get_user_input()
        response = self.agent.answer_query(prompt)
        await event_queue.enqueue_event(new_agent_text_message(response))

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        pass


def main() -> None:
    load_dotenv()
    host = os.environ.get("AGENT_HOST", "127.0.0.1")
    port = int(os.environ.get("WEB_RESEARCH_AGENT_PORT", "9997"))

    skill = AgentSkill(
        id="live_web_research",
        name="Live Web Research",
        description="Performs live web search and returns a concise answer with source citations.",
        tags=["research", "web_search", "citations", "faculty"],
        examples=[
            "What are common university policies for AI tool usage in assessments?",
            "Summarize recent guidance on academic integrity and generative AI.",
            "Find best practices for rubric design and cite sources.",
        ],
    )

    agent_card = AgentCard(
        name="WebResearchAgent",
        description="A live web-search research agent for university faculty. Always returns sources.",
        url=f"http://{host}:{port}/",
        version="1.0.0",
        default_input_modes=["text"],
        default_output_modes=["text"],
        capabilities=AgentCapabilities(streaming=False),
        skills=[skill],
    )

    request_handler = DefaultRequestHandler(
        agent_executor=WebResearchExecutor(),
        task_store=InMemoryTaskStore(),
    )

    server = A2AStarletteApplication(agent_card=agent_card, http_handler=request_handler)

    print(f"✅ Running WebResearchAgent at http://{host}:{port}/")
    uvicorn.run(server.build(), host=host, port=port)


if __name__ == "__main__":
    main()


Writing a2a_web_research_agent.py


### 8.1 Run the A2A server (Terminal)

```bash
python a2a_web_research_agent.py
```

Keep it running for the client test below.

Default URL: `http://127.0.0.1:9997/`


## 6. Call via A2A Client (discovery + send)

> Jupyter note: use top-level `await`.


In [8]:
import httpx
from IPython.display import Markdown, display

from a2a.client import ClientConfig, ClientFactory, create_text_message_object
from a2a.types import AgentCard, Artifact, Message, Task
from a2a.utils.message import get_message_text

def display_agent_card(agent_card: AgentCard) -> None:
    def esc(text: str) -> str:
        return str(text).replace("|", r"\|")
    md_parts = [
        "### Agent Card Details",
        "| Property | Value |",
        "| :--- | :--- |",
        f"| **Name** | {esc(agent_card.name)} |",
        f"| **Description** | {esc(agent_card.description)} |",
        f"| **Version** | `{esc(agent_card.version)}` |",
        f"| **URL** | `{esc(agent_card.url)}` |",
        f"| **Protocol Version** | `{esc(agent_card.protocol_version)}` |",
    ]
    display(Markdown("\n".join(md_parts)))

async def call_a2a_agent(prompt: str, host: str, port: int) -> str:
    async with httpx.AsyncClient(timeout=180.0) as httpx_client:
        client = await ClientFactory.connect(
            f"http://{host}:{port}",
            client_config=ClientConfig(httpx_client=httpx_client),
        )
        card = await client.get_card()
        display_agent_card(card)

        message = create_text_message_object(content=prompt)
        responses = client.send_message(message)

        text_content = ""
        async for response in responses:
            if isinstance(response, Message):
                text_content = get_message_text(response)
            elif isinstance(response, tuple):
                task: Task = response[0]
                if task.artifacts:
                    artifact: Artifact = task.artifacts[0]
                    text_content = get_message_text(artifact)
        return text_content

# Start the server in a terminal first:
#   python a2a_web_research_agent.py

prompt = "Summarize 3 best practices universities publish for student use of AI tools, with citations."
result = await call_a2a_agent(prompt, host="127.0.0.1", port=9997)
display(Markdown("## Final Response\n---\n" + (result or "*No response received*")))

### Agent Card Details
| Property | Value |
| :--- | :--- |
| **Name** | WebResearchAgent |
| **Description** | A live web-search research agent for university faculty. Always returns sources. |
| **Version** | `1.0.0` |
| **URL** | `http://127.0.0.1:9997/` |
| **Protocol Version** | `0.3.0` |

## Final Response
---
Universities have established several best practices to guide students in the responsible use of AI tools:

1. **Transparency and Documentation**: Students should openly disclose their use of AI tools in academic work. This includes citing AI assistance appropriately and documenting how AI was utilized, such as the specific tools and prompts used. This practice promotes academic integrity and allows educators to understand and assess the extent of AI involvement in student submissions. ([ox.ac.uk](https://www.ox.ac.uk/students/academic/guidance/skills/ai-study?utm_source=openai))

2. **Critical Evaluation of AI Outputs**: Students are encouraged to critically assess AI-generated content. This involves verifying the accuracy of information, identifying potential biases, and ensuring that the AI's output aligns with academic standards. Such critical engagement helps maintain the quality and credibility of academic work. ([privsec.harvard.edu](https://privsec.harvard.edu/best-practices-generative-ai-community-members?utm_source=openai))

3. **Adherence to Academic Integrity Policies**: Universities emphasize the importance of following institutional guidelines regarding AI use. Students should familiarize themselves with their institution's policies on AI tools, ensuring that their use aligns with ethical standards and does not compromise academic integrity. ([ox.ac.uk](https://www.ox.ac.uk/students/academic/guidance/skills/ai-study?utm_source=openai))

By adhering to these practices, students can effectively integrate AI tools into their learning while upholding the principles of academic integrity and critical thinking.

**Sources**:
- "Use of generative AI tools to support learning" — https://www.ox.ac.uk/students/academic/guidance/skills/ai-study
- "Best Practices: Generative AI (Community Members)" — https://privsec.harvard.edu/best-practices-generative-ai-community-members
- "Use of generative AI tools to support learning" — https://www.ox.ac.uk/students/academic/guidance/skills/ai-study

Sources:
- Source — https://teaching.charlotte.edu/teaching-support/teaching-guides/general-principles-teaching-age-ai?utm_source=openai
- Source — https://www.xavier.edu/teachingwithtech/genai/genai-bp?utm_source=openai
- Source — https://www.ox.ac.uk/students/academic/guidance/skills/ai-study?utm_source=openai
- Source — https://sc.edu/about/offices_and_divisions/cte/teaching_resources/generative_ai/genai_teaching_guidelines/index.php?utm_source=openai
- Source — https://privsec.harvard.edu/best-practices-generative-ai-community-members?utm_source=openai
- Source — https://cnu.atlassian.net/wiki/display/ckb/ai%2Busage%2Bbest%2Bpractices?utm_source=openai

Notes:
- (Auto-added sources from tool metadata.)

## 7. Optional: CrewAI version (custom tool calling OpenAI web_search)

This uses a custom tool, so you still only need your OpenAI key.

(If you run into version issues in workshops, skip this section.)


In [ ]:
# Optional CrewAI demo (only run if you installed crewai + crewai-tools)
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from agents_research import WebResearchAgent

@tool("OpenAI Web Search")
def openai_web_search(query: str) -> str:
    """Perform a live web search and return an answer with sources."""
    agent = WebResearchAgent(model="gpt-4o-mini")
    return agent.answer_query(query)

researcher = Agent(
    role="University Policy Researcher",
    goal="Research university-facing guidance and summarise with citations.",
    backstory="You help faculty quickly find up-to-date guidance and cite sources.",
    tools=[openai_web_search],
    allow_delegation=False,
    verbose=True,
)

task = Task(
    description="Research: {topic}. Provide 3-5 bullet best practices and cite sources.",
    expected_output="Answer + Sources + Notes",
    agent=researcher,
)

crew = Crew(agents=[researcher], tasks=[task], process=Process.sequential, verbose=True)
print(crew.kickoff(inputs={"topic": "academic integrity policies on generative AI"}))

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  9485fe9d-6f3a-4f87-b3e5-28f0e88257e0                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research: academic integrity policies on generative AI. Provide 3-5 bullet best practices and cite       │
│  sources.                                                                                                       │
│  ID: cff8764f-a88f-48d0-9dab-8b3b314d20d2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: University Policy Researcher                                                                            │
│                                                                                                                 │
│  Task: Research: academic integrity policies on generative AI. Provide 3-5 bullet best practices and cite       │
│  sources.                                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: open_ai_web_search                                                                                       │
│  Args: {'query': 'academic integrity policies generative AI 2024'}                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool open_ai_web_search executed with result: As of February 2026, academic institutions have been actively developing and implementing policies to address the challenges posed by generative AI technologies to academic integrity. These policies a...


...

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  9485fe9d-6f3a-4f87-b3e5-28f0e88257e0                                                                           │
│  Final Output: Best Practices for Academic Integrity Policies on Generative AI:                                 │
│                                                                                                                 │
│  - Clearly define permissible and impermissible uses of generative AI in academic work. For example, prohibit   │
│  completing assignments or exams solely by AI without personal input and require attribution when AI tools      │
│  contribute to content. (University of North Alabama, Wright State University)                                  │
│                                                                                                                 │
│  - Develop consistent, institution-wide policies while allowing departments or instructors to tailor            │
│  guidelines for specific courses or assessments to balance flexibility with uniformity. (Stanford University,   │
│  Carnegie Mellon University)                                                                                    │
│                                                                                                                 │
│  - Emphasize maintaining traditional academic integrity principles such as avoiding plagiarism, fabrication,    │
│  and cheating, while recognizing new challenges posed by AI-generated content. (West Chester University of      │
│  Pennsylvania)                                                                                                  │
│                                                                                                                 │
│  - Promote transparency by requiring students to disclose AI use in their work, fostering responsible and       │
│  ethical tool use rather than outright bans. (Wright State University, Carnegie Mellon University)              │
│                                                                                                                 │
│  - Provide educational resources and training for faculty and students on ethical AI integration and the        │
│  implications of AI-generated work on learning outcomes and academic honesty. (Duke University, California      │
│  Community Colleges)                                                                                            │
│                                                                                                                 │
│  Sources:                                                                                                       │
│                                                                                                                 │
│  - University of North Alabama, Generative AI Policy:                                                           │
│  https://www.una.edu/academics/generative-ai-policy.html?utm_source=openai                                      │
│  - Wright State University, Academic Integrity Standards:                                                       │
│  https://www.wright.edu/sites/www.wright.edu/files/uploads/2024/Feb/meeting/Policy3710-Tracked.pdf?utm_source=  │
│  openai                                               

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 8. Exercises (for faculty participants)

1) Add domain filters (advanced):
- In `tools=[{...}]`, use `filters.allowed_domains` to prioritize official sources (e.g., university websites)

2) Add a freshness instruction:
- Prefer sources from the last 12 months (unless citing older policy documents)

3) Add a “verify before publish” checklist:
- Return 3 checks faculty should verify before putting guidance in the LMS.
